In [4]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt
import pandas as pd

from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW
from einops import rearrange

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [5]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT-2 has no pad token → use EOS as padding
tokenizer.pad_token = tokenizer.eos_token


In [6]:
baseline_model = GPT2LMHeadModel.from_pretrained("gpt2")
baseline_model.to(device)
baseline_model.eval()


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [7]:
def kronecker_decompose(W, m, n, k=1):
    out_dim, in_dim = W.shape
    m2 = out_dim // m
    n2 = in_dim // n

    W_re = rearrange(
        W,
        '(m m2) (n n2) -> (m n) (m2 n2)',
        m=m, m2=m2, n=n, n2=n2
    )

    U, S, V = torch.svd_lowrank(W_re, q=k)

    A = rearrange(U, '(m n) k -> k m n', m=m, n=n)
    B = rearrange(V, '(m2 n2) k -> k m2 n2', m2=m2, n2=n2)

    scale = S.sqrt().view(-1, 1, 1)
    return A * scale, B * scale


In [8]:
def adaptive_normalize(W, A, B):
    W_hat = torch.kron(A[0], B[0])
    alpha = torch.norm(W, p="fro") / torch.norm(W_hat, p="fro")
    return A * torch.sqrt(alpha), B * torch.sqrt(alpha)


In [9]:
class KroneckerLinear(nn.Module):
    def __init__(self, A, B, bias):
        super().__init__()
        self.A = nn.Parameter(A)
        self.B = nn.Parameter(B)
        self.bias = nn.Parameter(bias.clone())

    def forward(self, x):
        y = torch.matmul(x, self.A)
        y = y.unsqueeze(-1) * self.B
        y = y.reshape(y.shape[0], y.shape[1], -1)
        return y + self.bias


In [10]:
class KroneckerLinearProj(nn.Module):
    def __init__(self, A, B, bias):
        super().__init__()
        self.A = nn.Parameter(A)
        self.B = nn.Parameter(B)
        self.bias = nn.Parameter(bias.clone())

    def forward(self, x):
        x = x.view(x.shape[0], x.shape[1], -1, 2)
        x = (x * self.B).sum(dim=-1)
        x = torch.matmul(x, self.A.t())
        return x + self.bias


In [11]:
def compress_one_layer(model, layer_idx):
    block = model.transformer.h[layer_idx]

    # c_fc
    W_fc = block.mlp.c_fc.weight
    b_fc = block.mlp.c_fc.bias

    A_fc, B_fc = kronecker_decompose(W_fc, m=768, n=1536)
    A_fc, B_fc = adaptive_normalize(W_fc, A_fc, B_fc)

    block.mlp.c_fc = KroneckerLinear(
        A_fc[0].to(device),
        B_fc[0].to(device),
        b_fc.to(device)
    )

    # c_proj
    W_proj = block.mlp.c_proj.weight.T
    b_proj = block.mlp.c_proj.bias

    A_p, B_p = kronecker_decompose(W_proj, m=768, n=1536)
    A_p, B_p = adaptive_normalize(W_proj, A_p, B_p)

    block.mlp.c_proj = KroneckerLinearProj(
        A_p[0].to(device),
        B_p[0].to(device),
        b_proj.to(device)
    )


In [12]:
compressed_model = GPT2LMHeadModel.from_pretrained("gpt2")
compressed_model.to(device)

for layer_idx in range(12):
    compress_one_layer(compressed_model, layer_idx)

compressed_model.eval()
print("12-layer compressed model ready.")


12-layer compressed model ready.


In [13]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")


In [14]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_dataset.set_format("torch")


In [15]:
train_loader = DataLoader(
    tokenized_dataset["train"],
    batch_size=8,
    shuffle=True
)

eval_loader = DataLoader(
    tokenized_dataset["validation"],
    batch_size=8
)


In [16]:
def evaluate_perplexity(model, dataloader):
    model.eval()
    total_loss = 0
    total_batches = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            labels = input_ids.clone()
            labels[attention_mask == 0] = -100

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_loss += outputs.loss.item()
            total_batches += 1

    avg_loss = total_loss / total_batches
    return math.exp(avg_loss)


In [17]:
ppl_before = evaluate_perplexity(compressed_model, eval_loader)
print("Compressed model perplexity (before training):", ppl_before)


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Compressed model perplexity (before training): 20905.43168001776


In [ ]:
# from torch.optim import AdamW

# optimizer = AdamW(compressed_model.parameters(), lr=3e-5)

# compressed_model.train()

# for epoch in range(2):   # increase to 2 epochs
#     total_loss = 0
#     steps = 0

#     for batch in train_loader:
#         input_ids = batch["input_ids"].to(device)
#         attention_mask = batch["attention_mask"].to(device)

#         labels = input_ids.clone()
#         labels[attention_mask == 0] = -100

#         optimizer.zero_grad()

#         outputs = compressed_model(
#             input_ids=input_ids,
#             attention_mask=attention_mask,
#             labels=labels
#         )

#         loss = outputs.loss
#         loss.backward()

#         torch.nn.utils.clip_grad_norm_(compressed_model.parameters(), 1.0)

#         optimizer.step()

#         total_loss += loss.item()
#         steps += 1

#     print(f"Epoch {epoch+1}, Avg Loss: {total_loss/steps}")



In [1]:
# ppl_after = evaluate_perplexity(compressed_model, eval_loader)
# print("Compressed model perplexity (after training):", ppl_after)


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("CUDA version:", torch.version.cuda)


CUDA available: False
GPU name: No GPU
CUDA version: None


In [3]:
import sys
print(sys.executable)


C:\Users\mahakisore\miniconda3\python.exe


In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)


CUDA available: True
GPU name: NVIDIA GeForce RTX 4050 Laptop GPU
CUDA version: 11.8
